# Lot Size and Water Use by Water-Access Category — replication

This notebook reproduces three companion charts comparing **lot size** and **water use** across
Bernalillo County residential parcels inside the ABCWUA drinking-water service area, split into
four categories by water access:

1. **ABCWUA only** — utility water, no well, no ditch/acequia access
2. **ABCWUA + well** — utility water plus a domestic well
3. **ABCWUA + ditch** — utility water plus ditch (acequia/MRGCD) access
4. **ABCWUA + well + ditch** — all three

All three metrics — median lot size, median effective ET per acre, and median total effective ET
per parcel — rise together across these four categories, roughly an 8x, 2.6x, and compounding 22x
spread respectively from utility-only lots to lots with every water source.

**Published figures** (`docs/water_access_categories/README.md`) — the versions presented in the
accompanying talk. This notebook reproduces the same methodology; a fresh run will differ
slightly if OSE POD, MRGCD ISOlog, or Bernalillo parcel data have been updated since the published
vintage (see "Notes and caveats" at the end).

## Method summary

- **Population:** Bernalillo County residential parcels (assessor `PROPCLASS = 'R'`) whose
  representative point falls inside the ABCWUA service-area boundary.
- **HAS_WELL:** >=1 active domestic-use OSE well (`DOM`/`DOL`/`PDM`/`DCN`, status `ACT`) inside the
  parcel; PLSS centroid-stack PODs dropped (same convention as the domestic-wells notebook).
- **HAS_ISOLOG ("ditch water"):** >=10% of parcel area overlaps the MRGCD 2021 ISOlog layer — a
  proxy for ditch/acequia access, not a literal water-right record.
- **Effective ET:** OpenET ensemble evapotranspiration minus 0.7 x GRIDMET precipitation, 2024,
  summed over each parcel's actual geometry via Google Earth Engine `reduceRegions` — a genuine
  per-parcel total.
- **Statistics:** median lot size, median effective ET per acre, and median total effective ET per
  parcel, for each of the four categories.


## Setup — Google Earth Engine account

Like the quadrant-map notebook, this pulls per-parcel evapotranspiration directly from **Google
Earth Engine (GEE)**. You'll need your own (free) GEE account and a Google Cloud project with the
Earth Engine API enabled:

1. **Sign up for Earth Engine** (no cost for research/non-commercial use):
   <https://earthengine.google.com/signup/>
2. **Create a Google Cloud project** and enable the Earth Engine API:
   <https://console.cloud.google.com/> -> *APIs & Services* -> enable "Google Earth Engine API".
   Or register the project directly at <https://code.earthengine.google.com/register>.
3. **Install the Python client** if needed: `pip install earthengine-api`.
4. **Replace the placeholder below** (`GEE_PROJECT`) with your own Google Cloud project ID.
5. Run the next cell — `ee.Authenticate()` opens a browser login the first time only.

**Requirements:** `earthengine-api`, `geopandas`, `pandas`, `requests`, `shapely`, `matplotlib`
(Python 3.10+).

**Runtime note:** Step 4 below (per-parcel effective ET) calls Earth Engine once per 500-parcel
chunk across roughly 190,000 parcels — around 380 network round trips. This can take tens of
minutes. The notebook prints progress as it goes.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import ee

# --- EDIT THIS: your own Google Cloud project ID with the Earth Engine API enabled ---
GEE_PROJECT = "your-gee-project-id"  # <-- replace with your project ID

ee.Authenticate()  # first run only: opens a browser login; cached afterward
ee.Initialize(project=GEE_PROJECT)
print("Earth Engine initialized with project:", GEE_PROJECT)


## Step 1 — Fetch Bernalillo County residential parcels (live, public, no download)

Parcel geometry and the assessor's residential classification come from the City of
Albuquerque / Bernalillo County shared GIS (CABQ AGIS) public ArcGIS REST feature service. No
account or download needed — the notebook pages through the service and keeps only residential
(`PROPCLASS = 'R'`) parcels.


In [ ]:
import geopandas as gpd
import pandas as pd
import requests

PARCELS_URL = (
    "https://services.arcgis.com/CWv1abTnC3urn4bV/arcgis/rest/services/"
    "berncoparcels_forIDO/FeatureServer/0/query"
)

def fetch_residential_parcels():
    """Page through the CABQ AGIS parcel feature service, keeping PROPCLASS='R' only."""
    meta = requests.get(PARCELS_URL.replace("/query", ""), params={"f": "json"}, timeout=30).json()
    page_size = meta.get("maxRecordCount", 1000)
    print(f"Service max page size: {page_size}")

    features = []
    offset = 0
    while True:
        resp = requests.get(PARCELS_URL, params={
            "where": "PROPCLASS='R'",
            "outFields": "UPC,PROPCLASS,VALCLASS,ACREAGE,SITUSADD",
            "returnGeometry": "true",
            "resultOffset": offset,
            "resultRecordCount": page_size,
            "f": "geojson",
        }, timeout=60)
        resp.raise_for_status()
        gj = resp.json()
        batch = gj.get("features", [])
        features.extend(batch)
        print(f"  fetched {len(features):,} parcels so far...")
        if len(batch) < page_size:
            break
        offset += page_size

    return gpd.GeoDataFrame.from_features(features, crs="EPSG:4326")

parcels = fetch_residential_parcels()
print(f"\nresidential parcels (Bernalillo): {len(parcels):,}")


## Step 2 — Obtain the three source shapefiles

Three shapefiles are needed and are **not committed** to this repository (see
`data/water_access_categories/README.md` for full download/request instructions):

| Zip file | Contents | Source |
|---|---|---|
| `OSE_PODs.zip` | OSE Points of Diversion | Public — NM OSE geospatial open-data portal |
| `ABCWUAServiceArea.zip` | ABCWUA service-area boundary | Public — NM OSE PWS boundaries |
| `MRGCD_ISOlog.zip` | MRGCD 2021 ISOlog (ditch-access proxy) | **Obtained from MRGCD** — request from `mapping@mrgcd.us`, not on a public portal |

Place all three zips in `data/water_access_categories/input/source_shapefiles/` before running
the next cell.


In [ ]:
import zipfile
from pathlib import Path

def find_package_root(start=None):
    cur = Path(start or Path.cwd()).resolve()
    for p in [cur, *cur.parents]:
        if (p / "data" / "water_access_categories").is_dir():
            return p
    raise FileNotFoundError(
        "Could not find the package root (containing data/water_access_categories) above the notebook.")

PKG_ROOT = find_package_root()
WAC_DIR = PKG_ROOT / "data" / "water_access_categories"
SRC_ZIP_DIR = WAC_DIR / "input" / "source_shapefiles"
WORK_DIR = WAC_DIR / "input" / "working_unzipped"
WORK_DIR.mkdir(parents=True, exist_ok=True)

ARCHIVES = {
    "OSE_PODs.zip":            ("OSE_PODs",            "Points_of_Diversion.shp"),
    "ABCWUAServiceArea.zip":   ("ABCWUAServiceArea",   "ABCWUAServiceArea.shp"),
    "MRGCD_ISOlog.zip":        ("MRGCD_ISOlog",        "MRGCDISOLogs2021.shp"),
}

shp_paths = {}
for archive, (subdir, shp_name) in ARCHIVES.items():
    zip_path = SRC_ZIP_DIR / archive
    if not zip_path.exists():
        raise FileNotFoundError(
            f"Missing source archive: {zip_path}\n"
            f"See data/water_access_categories/README.md for where to obtain and place it."
        )
    out_dir = WORK_DIR / subdir
    if not out_dir.exists():
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(out_dir)
    shp_paths[archive] = out_dir / shp_name
    print(f"{archive} -> {shp_paths[archive]}")

POD_SHP = shp_paths["OSE_PODs.zip"]
ABCWUA_SHP = shp_paths["ABCWUAServiceArea.zip"]
ISOLOG_SHP = shp_paths["MRGCD_ISOlog.zip"]


## Step 3 — Classify parcels into four water-access categories

1. Reproject everything to UTM Zone 13N (EPSG:26913, meters) for area math.
2. Keep parcels whose **representative point** falls inside the ABCWUA boundary.
3. **HAS_WELL:** join active domestic-use OSE wells (`DOM`/`DOL`/`PDM`/`DCN`, status `ACT`) to
   parcels; drop PLSS centroid-stack wells first (>=10 wells sharing one rounded coordinate with
   no quarter-quarter section code) so an artifact of imprecise well locations doesn't get counted
   as multiple distinct wells.
4. **HAS_ISOLOG:** compute the fraction of each parcel's area that overlaps the MRGCD ISOlog
   layer; flag `True` where that fraction is >= 10%.
5. Combine both flags into the four-category label.


In [ ]:
import numpy as np

UTM13 = "EPSG:26913"
ISO_FRAC_MIN = 0.10
STACK_THRESHOLD = 10

parcels_utm = parcels.to_crs(UTM13).reset_index(drop=True)
parcels_utm["pid"] = parcels_utm.index

abcwua = gpd.read_file(ABCWUA_SHP).to_crs(UTM13)
abcwua_u = abcwua.union_all()
rp = parcels_utm.geometry.representative_point()
inside = rp.within(abcwua_u)
sub = parcels_utm[inside].copy().reset_index(drop=True)
print(f"inside ABCWUA service area: {len(sub):,} of {len(parcels_utm):,} residential parcels")

# ---- well flag ----
pods = gpd.read_file(POD_SHP,
    columns=["pod_status", "use_", "qtr_4th", "pod_file"],
    where="pod_status='ACT' AND use_ IN ('DOM','DOL','PDM','DCN')").to_crs(UTM13)
xy = np.column_stack([pods.geometry.x.round(1), pods.geometry.y.round(1)])
ck = pd.Series(map(tuple, xy), index=pods.index)
stk = ck.map(ck.value_counts())
qtr = pods["qtr_4th"].astype("string").str.strip().replace("", pd.NA)
pods = pods[~(qtr.isna() & (stk >= STACK_THRESHOLD))].copy()

jw = gpd.sjoin(pods, sub[["pid", "geometry"]], predicate="within", how="inner")
well_ct = jw.groupby("pid").size()
sub["WELL_N"] = sub["pid"].map(well_ct).fillna(0).astype(int)
sub["HAS_WELL"] = sub["WELL_N"] > 0
print(f"parcels with a well: {int(sub['HAS_WELL'].sum()):,}")

# ---- ISOlog ("ditch water") overlap fraction ----
iso = gpd.read_file(ISOLOG_SHP).to_crs(UTM13)
sub["parea"] = sub.geometry.area
ov = gpd.overlay(sub[["pid", "parea", "geometry"]], iso[["geometry"]], how="intersection")
ov["iarea"] = ov.geometry.area
isofrac = ov.groupby("pid")["iarea"].sum() / sub.set_index("pid")["parea"]
sub["ISO_FRAC"] = sub["pid"].map(isofrac).fillna(0.0)
sub["HAS_ISOLOG"] = sub["ISO_FRAC"] >= ISO_FRAC_MIN
print(f"parcels with ditch (ISOlog) access: {int(sub['HAS_ISOLOG'].sum()):,}")

# ---- categorize ----
def categorize(row):
    if row.HAS_WELL and row.HAS_ISOLOG:
        return "4 ABCWUA+well+ISOlog"
    if row.HAS_WELL:
        return "2 ABCWUA+well"
    if row.HAS_ISOLOG:
        return "3 ABCWUA+ISOlog"
    return "1 ABCWUA only"

sub["CATEGORY"] = sub.apply(categorize, axis=1)
sub["ACRES"] = sub["parea"] / 4046.856

print("\n=== category counts and lot size ===")
summary = sub.groupby("CATEGORY").agg(
    n=("pid", "size"), median_acres=("ACRES", "median"))
print(summary.round(3).to_string())


## Step 4 — Per-parcel 2024 effective ET (Google Earth Engine)

For each classified parcel, sum 2024 effective ET (OpenET ensemble minus 0.7 x GRIDMET
precipitation) over its actual geometry using Earth Engine's `reduceRegions`, in 500-parcel
chunks. This produces a genuine per-parcel total in acre-feet — not an approximation from a
mean depth times area.

**This is the slow step** — expect tens of minutes for ~190,000 parcels.


In [ ]:
YEAR = 2024
M3_AF = 0.000810713194
CHUNK = 500

et_collection = (ee.ImageCollection("OpenET/ENSEMBLE/CONUS/GRIDMET/MONTHLY/v2_0")
                  .filterDate(f"{YEAR}-01-01", f"{YEAR + 1}-01-01"))
precip_annual_07 = (ee.ImageCollection("IDAHO_EPSCOR/GRIDMET")
                    .filterDate(f"{YEAR}-01-01", f"{YEAR + 1}-01-01")
                    .select("pr").sum().multiply(0.7))

effective_et_volume = (
    et_collection.select("et_ensemble_mad").sum()
    .subtract(precip_annual_07).max(0)
    .divide(1000).multiply(ee.Image.pixelArea())
    .rename("eff_et_m3")
)

sub_wgs84 = sub.to_crs("EPSG:4326")
records = {}
for i in range(0, len(sub_wgs84), CHUNK):
    part = sub_wgs84.iloc[i:i + CHUNK]
    feats = [ee.Feature(ee.Geometry(geom.__geo_interface__), {"pid": int(pid)})
              for pid, geom in zip(part["pid"], part.geometry)]
    result = effective_et_volume.reduceRegions(
        collection=ee.FeatureCollection(feats), reducer=ee.Reducer.sum(),
        scale=30, tileScale=4)
    for f in result.getInfo()["features"]:
        props = f["properties"]
        records[props["pid"]] = props.get("eff_et_m3", 0.0)
    print(f"  ET {min(i + CHUNK, len(sub_wgs84)):,}/{len(sub_wgs84):,}")

et_df = pd.DataFrame(list(records.items()), columns=["pid", "eff_et_m3"])
sub = sub.merge(et_df, on="pid", how="left")
sub["eff_et_m3"] = sub["eff_et_m3"].fillna(0.0)
sub["Q_MID_AF"] = sub["eff_et_m3"] * M3_AF
print(f"\ntotal 2024 effective ET: {sub['Q_MID_AF'].sum():,.0f} acre-feet")


## Step 5 — Three summary charts

Same four categories throughout: median lot size, median effective ET per acre (normalizes out
lot-size differences), and median total effective ET per parcel (the actual volume on the median
parcel — see the caveat below on why this isn't simply the other two medians multiplied
together).


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimage
import matplotlib.font_manager as fm
import matplotlib.ticker as mticker

CAT_ORDER = ["1 ABCWUA only", "2 ABCWUA+well", "3 ABCWUA+ISOlog", "4 ABCWUA+well+ISOlog"]
X_LABELS = ["ABCWUA\nonly", "ABCWUA\n+ well", "ABCWUA\n+ ditch", "ABCWUA + well\n+ ditch"]
SLATE = "#0f172a"
GREY = "#5a5a5a"
BAR_COLORS = ["#94a3b8", "#5eead4", "#0f766e", "#0f172a"]
PALE_GREY = "#E8E8E8"
FONT = "Helvetica" if "Helvetica" in {f.name for f in fm.fontManager.ttflist} else "Arial"
plt.rcParams["font.family"] = FONT

sub["ET_AF_AC"] = sub["Q_MID_AF"] / sub["ACRES"]

def category_bar_chart(value_col, title, subtitle, ylabel, value_fmt, ymax, out_name):
    g = sub.groupby("CATEGORY").agg(n=("pid", "size"), value=(value_col, "median")).reindex(CAT_ORDER)
    values = g["value"].tolist()
    counts = g["n"].astype(int).tolist()

    fig, ax = plt.subplots(figsize=(10.5, 7))
    fig.subplots_adjust(left=0.09, right=0.97, top=0.82, bottom=0.14)
    x = range(len(CAT_ORDER))
    ax.bar(x, values, color=BAR_COLORS, width=0.62, zorder=3, edgecolor="white", linewidth=0.8)
    for i, (v, n) in enumerate(zip(values, counts)):
        ax.text(i, v + ymax * 0.012, f"{value_fmt.format(v)}\n(n={n:,})", ha="center", va="bottom",
                fontsize=12.5, fontweight="bold", color=SLATE, zorder=5, linespacing=1.25)

    fig.text(0.5, 0.935, title, ha="center", va="center", fontsize=20, fontweight="bold", color=SLATE)
    fig.text(0.5, 0.883, subtitle, ha="center", va="center", fontsize=13, color=GREY)
    ax.set_ylabel(ylabel, fontsize=12, color=SLATE)
    ax.set_xticks(list(x))
    ax.set_xticklabels(X_LABELS, fontsize=12.5, color=SLATE)
    ax.set_ylim(0, ymax)
    ax.tick_params(colors=SLATE, labelsize=11)
    ax.grid(axis="y", color=PALE_GREY, linewidth=0.9, zorder=0)
    ax.set_axisbelow(True)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    for spine in ["left", "bottom"]:
        ax.spines[spine].set_color(SLATE)

    logo_path = PKG_ROOT / "docs" / "logo.png"
    if logo_path.exists():
        logo = mpimage.imread(logo_path)
        logo_h = 0.052
        logo_w = logo_h * (logo.shape[1] / logo.shape[0]) * (7 / 10.5)
        lax = fig.add_axes([0.075, 0.015, logo_w, logo_h], anchor="SW", zorder=10)
        lax.imshow(logo); lax.axis("off")
    fig.text(0.97, 0.03, "Data: Bernalillo County Assessor, OSE POD, OSE ABCWUA boundary, "
             "MRGCD ISOlog, OpenET, GRIDMET", ha="right", va="bottom", fontsize=8.5, color=GREY)

    out_path = OUT_DIR / out_name
    fig.savefig(out_path, dpi=200)
    plt.show()
    print(f"Wrote {out_path}")
    for lbl, v, n in zip(CAT_ORDER, values, counts):
        print(f"  {lbl:24s} n={n:>7,}  median={value_fmt.format(v)}")

OUT_DIR = WAC_DIR / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

category_bar_chart("ACRES", "Lot Size by Water Access, ABCWUA Service Area",
    "Median parcel size, Bernalillo County residential parcels",
    "Median lot size (acres)", "{:.2f} ac", 1.6, "lot_size_by_category.png")


In [ ]:
category_bar_chart("ET_AF_AC", "Effective ET by Water Access, ABCWUA Service Area",
    "Median effective ET, Bernalillo County residential parcels, 2024",
    "Median effective ET (acre-feet per acre / year)", "{:.2f} AF/ac", 2.3,
    "effective_et_by_category.png")


In [ ]:
category_bar_chart("Q_MID_AF", "Total Water Use by Water Access, ABCWUA Service Area",
    "Median total effective ET per parcel, Bernalillo County residential parcels, 2024",
    "Median total effective ET (acre-feet/year)", "{:.2f} AF/yr", 3.0,
    "total_water_use_by_category.png")


## Compare to the published figures

The published talk figures are included in this repository at
`docs/water_access_categories/lot_size_by_category_full_abcwua.png`,
`docs/water_access_categories/effective_et_by_category_full_abcwua.png`, and
`docs/water_access_categories/total_water_use_by_category_full_abcwua.png` for direct comparison.


## Notes and caveats

- **"Ditch water" = ISOlog overlap, not a literal water-right record.** The MRGCD ISOlog layer
  maps irrigation-serviceable land, not confirmed acequia shares or actual diversions.
- **Category membership is a snapshot** (2021 ISOlog, current OSE POD vintage) — a parcel that
  had a well drilled since, or whose ditch access changed, is classified by current-data state.
- **`median(A) x median(B)` does not equal `median(A x B)`.** The total-water-use chart uses each
  parcel's own true per-parcel effective-ET total (from Earth Engine), not the lot-size median
  multiplied by the per-acre-ET median — those two quantities can differ by up to roughly 10% in
  the middle categories because lot size and per-acre ET intensity aren't perfectly correlated
  across parcels. Category 3 (ditch access, no well) showed the largest gap in the published run:
  the largest ditch-adjacent parcels tend to run somewhat lower per-acre ET than smaller ones,
  plausibly reflecting more fallow/pasture land diluting the average.
- **Parcel-scale resolution.** Most residential lots here (median 0.16-1.3 acres) are one or two
  OpenET/Landsat pixels (~30 m) or smaller, so an individual parcel's ET value carries real
  pixel-mixing noise. The category-level medians (thousands of parcels per bar, except category
  4's few hundred) are far more robust than any single parcel's number.
- **The ABCWUA boundary is provisional** — the OSE Public Water System polygon used here and
  ABCWUA's own service-zone map disagree by roughly 1,200 domestic rights. See
  `data/domestic_wells/README.md` for the same open boundary question.
- **The 0.7x effective-precipitation coefficient** follows the FAO rule-of-thumb effective-rainfall
  fraction for arid regions (Allen et al., 1998; Brouwer & Heibloem, 1986) — see
  `docs/quadrant_map/README.md` for the full citation and caveats. Not a locally-calibrated
  physical constant.
- **2024 cross-section only** — this is a snapshot, not a claim about how these categories are
  changing over time.
- **Residential parcels only** (assessor `PROPCLASS = 'R'`) — commercial, vacant, and other
  non-residential parcel classes are excluded from all four categories.
- **A fresh run will not exactly match the published figures** — OSE POD is updated monthly,
  Bernalillo County parcel data is served live and may have changed, and the MRGCD ISOlog vintage
  you obtain may differ from the 2021 layer used in the published run.
